# Phase 1: bigP3BCI EDF Inspection and Initial Preprocessing

本阶段的目标是确认 EDF 可以读取，了解 EEG channel、sampling rate、annotations / stimulus markers、target/non-target 标签结构，并进行初步、可视化的信号质量检查。正式 preprocessing 参数将在确认 lab 代码后确定；当前仅使用论文中的探索性设置。

本 notebook 不训练 SWLDA、EEGNet 或任何分类模型。

## 1. Setup

导入依赖，定位项目根目录，并准备报告输出目录。此处不下载或复制 EDF 文件。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

print(f'MNE version: {mne.__version__}')

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'README.md').exists() or (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError(
        f'Could not find project root from {start}. Start Jupyter inside the repository.'
    )
PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data directory: {RAW_DIR}')
print(f'Reports directory: {REPORT_DIR}')

MNE version: 1.12.1
Project root: D:\BCI_project\bigp3bci-personalized-strategies
Raw data directory: D:\BCI_project\bigp3bci-personalized-strategies\data\raw
Reports directory: D:\BCI_project\bigp3bci-personalized-strategies\outputs\reports


## 2. Select one participant and EDF files

先只使用一位 Study L ALS 用户的一份 Train EDF 和匹配的 Test EDF。请填写下面三个参数；路径可以是绝对路径，也可以是相对于项目根目录的路径。

In [ ]:
SUBJECT_ID = "..."
TRAIN_EDF = Path("...")
TEST_EDF = Path("...")

def resolve_input_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path

train_path = resolve_input_path(TRAIN_EDF)
test_path = resolve_input_path(TEST_EDF)
missing = [str(p) for p in (train_path, test_path) if not p.is_file()]
if SUBJECT_ID == '...' or '...' in (str(TRAIN_EDF), str(TEST_EDF)):
    print('请先填写 SUBJECT_ID、TRAIN_EDF 和 TEST_EDF，然后从本节重新运行。')
elif missing:
    print('以下 EDF 文件不存在，暂不继续读取：')
    for item in missing: print(f'  - {item}')
else:
    print(f'Subject: {SUBJECT_ID}')
    print(f'Train: {train_path}')
    print(f'Test:  {test_path}')

## 3. Read EDF files

仅在两个路径都有效时读取 EDF；不立即滤波，也不改变原始数据。

In [ ]:
if 'missing' not in globals() or missing or SUBJECT_ID == '...':
    print('跳过读取：请先完成上一节的 EDF 路径设置。')
else:
    raw_train = mne.io.read_raw_edf(train_path, preload=False, infer_types=True, verbose=False)
    raw_test = mne.io.read_raw_edf(test_path, preload=False, infer_types=True, verbose=False)
    def print_raw_summary(label, raw):
        print(f'\n{label}')
        print(f'  sampling rate: {raw.info["sfreq"]} Hz')
        print(f'  duration: {raw.n_times / raw.info["sfreq"]:.2f} s')
        print(f'  channels ({len(raw.ch_names)}): {raw.ch_names}')
        print(f'  inferred types: {dict(zip(raw.ch_names, raw.get_channel_types()))}')
        print(f'  annotations: {len(raw.annotations)}')
    print_raw_summary('Train EDF', raw_train)
    print_raw_summary('Test EDF', raw_test)

## 4. Channel inventory

下面分别列出 channel name、MNE 推断的 channel type 和可获取的 unit。论文中的 7 个 channel 仅作提示，不因大小写或命名差异自动判定失败。

In [ ]:
EXPECTED_EEG_CHANNELS = ['Fz', 'Cz', 'P3', 'Pz', 'P4', 'PO7', 'PO8']
def channel_inventory(raw):
    rows = []
    for name, kind, unit in zip(raw.ch_names, raw.get_channel_types(), raw._orig_units.values()):
        rows.append({'channel name': name, 'channel type': kind, 'unit': unit})
    return pd.DataFrame(rows)

if 'raw_train' in globals():
    train_inventory = channel_inventory(raw_train)
    test_inventory = channel_inventory(raw_test)
    display(train_inventory)
    display(test_inventory)
    for label, raw in [('Train', raw_train), ('Test', raw_test)]:
        present = [x for x in EXPECTED_EEG_CHANNELS if x in raw.ch_names]
        missing_expected = [x for x in EXPECTED_EEG_CHANNELS if x not in raw.ch_names]
        print(f'{label} expected channels present: {present}')
        print(f'{label} expected channels not exact-name matches: {missing_expected}')
        print('提示：请人工确认大小写、前缀或其他命名方式是否对应。')

## 5. Event and annotation inspection

先观察 annotations 的原始 description 和频数，不预设 event code。

**在确认 `StimulusType` 的实际存储方式和 target/non-target event code 之前，不进行 epoch 或标签映射。**

In [ ]:
def annotation_table(raw):
    return pd.DataFrame({'onset': raw.annotations.onset, 'duration': raw.annotations.duration, 'description': raw.annotations.description})
marker_words = ('stimulus', 'event', 'target', 'trigger', 'status', 'marker')
if 'raw_train' in globals():
    for label, raw in [('Train', raw_train), ('Test', raw_test)]:
        ann = annotation_table(raw)
        print(f'\n{label} annotations (first 20):')
        display(ann.head(20))
        if ann.empty:
            print('Annotations 为空；下一步需要检查 EDF 中的 event/stimulus channel。')
        else:
            print('Description counts:')
            display(ann['description'].value_counts())
            print('All unique descriptions:', ann['description'].unique().tolist())
        possible = [name for name in raw.ch_names if any(word in name.lower() for word in marker_words)]
        print(f'Possible marker/stimulus channels: {possible or "none found"}')

## 6. Raw EEG visualization

优先使用 7 个论文 channel 中实际存在的通道；否则使用 MNE 识别出的 EEG channels。每幅图显示前 10 秒原始波形。

In [ ]:
def choose_eeg_channels(raw):
    preferred = [x for x in EXPECTED_EEG_CHANNELS if x in raw.ch_names]
    return preferred or mne.pick_types(raw.info, eeg=True, exclude=[]).tolist()
def plot_raw_10s(label, raw):
    picks = choose_eeg_channels(raw)
    if not picks:
        print(f'{label}: 找不到 EEG channels，跳过波形图。')
        return
    data, times = raw.copy().pick(picks).get_data(start=0, stop=min(raw.n_times, int(10 * raw.info['sfreq'])), return_times=True)
    fig, axes = plt.subplots(len(picks), 1, sharex=True, figsize=(12, max(2, 1.5 * len(picks))))
    axes = np.atleast_1d(axes)
    for ax, signal, name in zip(axes, data, picks): ax.plot(times, signal); ax.set_ylabel(name); ax.grid(alpha=.3)
    axes[-1].set_xlabel('Time (s)'); fig.suptitle(f'{label} raw EEG: first 10 seconds'); fig.tight_layout(); plt.show()
if 'raw_train' in globals(): plot_raw_10s('Train', raw_train)
if 'raw_test' in globals(): plot_raw_10s('Test', raw_test)

## 7. Power spectral density (PSD) QC

PSD 范围为 0.5–80 Hz，用于观察频谱和可能的 60 Hz line noise。图片保存到 subject-specific report 目录。

In [ ]:
subject_report_dir = REPORT_DIR / str(SUBJECT_ID)
subject_report_dir.mkdir(parents=True, exist_ok=True) if SUBJECT_ID != '...' else None
def plot_psd(label, raw, filename):
    picks = choose_eeg_channels(raw)
    if not picks: print(f'{label}: 找不到 EEG channels，跳过 PSD。'); return
    spectrum = raw.compute_psd(fmin=0.5, fmax=80, picks=picks, reject_by_annotation=False, verbose=False)
    fig = spectrum.plot(average=True, show=False); fig.suptitle(f'{label} raw EEG PSD (0.5–80 Hz; inspect 60 Hz)')
    fig.savefig(subject_report_dir / filename, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)
if 'raw_train' in globals(): plot_psd('Train', raw_train, 'train_raw_psd.png')
if 'raw_test' in globals(): plot_psd('Test', raw_test, 'test_raw_psd.png')

## 8. Exploratory filtering — do not overwrite raw data

以下是 Gupta et al. 2026 描述的探索性设置：60 Hz notch 和 0.5–30 Hz band-pass。reference、bad-channel / artifact handling 和精确参数仍待 lab 确认。过滤仅作用于副本，`raw_train` / `raw_test` 保持不变。

In [ ]:
def exploratory_filter(raw):
    filtered = raw.copy()
    picks = choose_eeg_channels(filtered)
    if not picks: return filtered, []
    filtered.notch_filter(freqs=[60], picks=picks, verbose=False)
    filtered.filter(l_freq=0.5, h_freq=30, picks=picks, verbose=False)
    return filtered, picks
if 'raw_train' in globals():
    raw_train_filtered, _ = exploratory_filter(raw_train)
    raw_test_filtered, _ = exploratory_filter(raw_test)
    plot_raw_10s('Train filtered', raw_train_filtered)
    plot_raw_10s('Test filtered', raw_test_filtered)
    plot_psd('Train filtered', raw_train_filtered, 'train_filtered_psd.png')
    plot_psd('Test filtered', raw_test_filtered, 'test_filtered_psd.png')

## 9. Phase-1 checklist

请人工确认：

- [ ] Train/Test EDF 都能读取
- [ ] sampling rate 是否为 256 Hz
- [ ] 7 个论文 EEG channels 是否存在
- [ ] `StimulusType` / target / non-target 的实际编码是什么
- [ ] event 位于 annotations 还是专门的 channel
- [ ] Train/Test 的 event structure 是否一致
- [ ] 是否存在显著 60 Hz noise、坏通道或异常记录

只有完成上述确认后，下一 notebook 才实现 stimulus-locked `0–800 ms` epochs 和 target/non-target 标签提取。